# 00 — Bronze Inventory

**Objetivo:** establecer qué snapshots existen realmente en Bronze, cuánto pesan y cuántas filas/columnas contienen.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Están presentes exactamente las siete fuentes esperadas?
- ¿Cuál es el tamaño real de cada snapshot local?
- ¿Cuántas filas y columnas tiene cada CSV descargado?
- ¿Alguna fuente necesita fallback de parsing?

**Guardrail:** `data/bronze` es inmutable. Este notebook solo lee.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.profiling.inventory import build_inventory
from argos.common.reports import write_report

profiling_cfg = PROJECT_CONFIG["profiling"]

inventory = build_inventory(
    FILES,
    DATASETS,
    sample_size=profiling_cfg["csv_sample_size"],
    compute_sha256=profiling_cfg["compute_sha256"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

report_path = write_report(
    inventory,
    paths.reports / "profiling" / "dataset_inventory.parquet",
)

inventory[[
    "dataset", "source_id", "filename", "file_size_gb",
    "row_count", "column_count", "parse_mode", "elapsed_seconds"
]]

,dataset,source_id,filename,file_size_gb,row_count,column_count,parse_mode,elapsed_seconds
0,additions,cb9c-h8sn,SECOP_II_-_Adiciones_20260921.csv,6.830472,28378604,5,python_csv_streaming,70.750516
1,bidders,hgi6-6wh3,Proponentes_por_Proceso_SECOP_II_20260921.csv,0.578623,2316965,9,python_csv_streaming,5.195674
2,contracts,jbjy-vk9h,SECOP_II_-_Contratos_Electrónicos_20260921.csv,10.073206,6066730,85,python_csv_streaming,175.698406
3,payments,uymx-8p3j,SECOP_II_-_Plan_de_pagos_20260921.csv,0.000575,1000,34,python_csv_streaming,0.018756
4,processes,p6dx-8zbt,SECOP_II_-_Procesos_de_Contratación_20260921.csv,9.485029,9204423,59,python_csv_streaming,207.718400
5,sanctions,it5q-hg94,SECOPII_-_Multas_y_Sanciones_20260921.csv,0.000168,552,17,python_csv_streaming,0.007085
6,suppliers,qmzu-gj57,SECOP_II_-_Proveedores_Registrados_20260921.csv,0.592402,1612550,26,python_csv_streaming,5.913965


## Criterio de salida
No continúes si falta una fuente, si hay dos snapshots activos de la misma fuente o si un archivo no puede parsearse de manera reproducible.

**Decisión a documentar:** snapshot de Bronze que será la base de M1.